# 1. Import Libraries

In [50]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 2. Load Processed Dataset

In [51]:
data_path = "../data/processed/cleaned_news.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (44689, 7)


,title,text,subject,date,label,content,clean_text
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0,Ben Stein Calls Out 9th Circuit Court: Committ...,ben stein calls out 9th circuit court committe...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1,Trump drops Steve Bannon from National Securit...,trump drops steve bannon from national securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1,Puerto Rico expects U.S. to lift Jones Act shi...,puerto rico expects us to lift jones act shipp...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0,OOPS: Trump Just Accidentally Confirmed He Le...,oops trump just accidentally confirmed he leak...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1,Donald Trump heads for Scotland to reopen a go...,donald trump heads for scotland to reopen a go...


# 3. Feature Engineering

## Data Validation

In [52]:
# Make sure clean_text has no missing values
df["clean_text"] = df["clean_text"].fillna("")

# Convert everything into string
df["clean_text"] = df["clean_text"].astype(str)

# Remove empty text
df = df[df["clean_text"].str.strip() != ""]

print("\nAfter validation:")
print(df.shape)

print("\nData type check:")
print(df["clean_text"].apply(type).value_counts())


After validation:
(44680, 7)

Data type check:
clean_text
<class 'str'>    44680
Name: count, dtype: int64


## Split Train and Test Dataset

In [53]:
X = df["clean_text"].tolist()
y = df["label"].values

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 35744
Testing samples: 8936


## Tokenization

In [54]:
MAX_WORDS = 20000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Learn vocabulary only from training data
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index)

print("\nVocabulary size:")
print(vocab_size)


Vocabulary size:
223892


## Text to Sequences

In [55]:
X_train_sequences = tokenizer.texts_to_sequences(X_train)

X_test_sequences = tokenizer.texts_to_sequences(X_test)

print("\nExample original text:")
print(X_train[0])

print("\nExample sequence:")
print(X_train_sequences[0])


Example original text:
the real donald trump why america is rooting for this outsider video this video takes you from the beginning of trump s improbable rise in the ranks of professional politicians to the last man standing in the gop here s why america is rooting for trump

Example sequence:
[2, 384, 70, 15, 222, 156, 12, 14469, 10, 28, 6804, 113, 28, 113, 1075, 43, 26, 2, 1616, 4, 15, 11, 18432, 1386, 7, 2, 3948, 4, 2788, 1072, 3, 2, 96, 241, 1204, 7, 2, 429, 190, 11, 222, 156, 12, 14469, 10, 15]


In [56]:
# Analyze the distribution of sequence lengths in the training data
sequence_lengths = [
    len(seq)
    for seq in X_train_sequences
]


length_stats = pd.Series(sequence_lengths).describe()


print("\nSequence length statistics:")
print(length_stats)


Sequence length statistics:
count    35744.000000
mean       418.306009
std        354.508051
min          4.000000
25%        216.000000
50%        374.000000
75%        526.000000
max       8135.000000
dtype: float64


## Padding

In [57]:
MAX_LENGTH = 300

X_train_pad = pad_sequences(
    X_train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("\nPadded shape:")
print("X_train:", X_train_pad.shape)
print("X_test :", X_test_pad.shape)


Padded shape:
X_train: (35744, 300)
X_test : (8936, 300)


## Save Tokenizer

In [ ]:
os.makedirs("../models", exist_ok=True)

tokenizer_path = "../models/tokenizer.pkl"

with open(tokenizer_path, "wb") as file:
    pickle.dump(tokenizer,file)

print("\nTokenizer saved:")
print(tokenizer_path)


Tokenizer saved:
../models/tokenizer.pkl


## Summary

In [59]:
print("X_train shape:", X_train_pad.shape)

print("X_test shape:", X_test_pad.shape)

print("y_train shape:", y_train.shape)

print("y_test shape:", y_test.shape)

X_train shape: (35744, 300)
X_test shape: (8936, 300)
y_train shape: (35744,)
y_test shape: (8936,)
